# 04 — WhatsApp Collections Customer Canonicalization

## Objective

Objetivo

Transformar whatsapp_collections_history.csv — originalmente no grão de 1 linha por cliente

- engenharia de atributos orientada a Collections : inadimplência e evolução do atraso, exposição financeira, pressão e frequência de contatos, engajamento, momento do contato (timing), pagamentos , jornada e sequências de interação

Guardrail causal: associações observadas não representam necessariamente efeitos causais. Variáveis relacionadas a template, horário/momento do envio e características das interações descrevem associações históricas e exposição à política de cobrança vigente. Portanto, não devem ser interpretadas como efeito incremental do tratamento sem um desenho causal apropriado, como experimento randomizado ou metodologia causal equivalente.

Guardrail monetário: Amount_paid_brl representa um pagamento atribuído à janela de 72 horas após cada envio. As somas no nível do cliente podem ser preservadas como pagamento atribuído (attributed payment), mas devem ser reconciliadas com a trajetória do saldo e com possíveis sobreposições das janelas de atribuição antes de serem interpretadas como recuperação econômica efetiva.

In [2]:
from pathlib import Path
import ast
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

# Works both in the local repository and with the attached case file.
CANDIDATES = [
    Path("../data/raw/whatsapp_collections_history.csv"),
    Path("../data/whatsapp_collections_history.csv"),
    Path("/mnt/data/whatsapp_collections_history(1).csv"),
]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("whatsapp_collections_history.csv not found.")

OUTPUT_DIR = Path("../data/interim") if Path("../data").exists() else Path("/mnt/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input :", DATA_PATH.resolve())
print("Output:", OUTPUT_DIR.resolve())

Input : C:\Users\beelt\Documents\collections_case_candidate\data\raw\whatsapp_collections_history.csv
Output: C:\Users\beelt\Documents\collections_case_candidate\data\interim


## 4.1 Load and source contract

In [3]:
whatsapp = pd.read_csv(DATA_PATH, parse_dates=["sent_at"])

required = {
    "message_id","customer_id","sent_at","template","n_msgs_last_14d",
    "days_past_due","outstanding_balance_brl","monthly_salary_brl",
    "payday_day_of_month","n_prior_transactions","account_age_months",
    "days_since_last_app_login","state_uf","delivery_status","interaction",
    "paid_within_72h","amount_paid_brl"
}
missing = sorted(required - set(whatsapp.columns))
assert not missing, f"Missing required columns: {missing}"

assert whatsapp["message_id"].notna().all()
assert whatsapp["message_id"].is_unique, "message_id must be unique at source grain."
assert whatsapp["customer_id"].notna().all()

print("Rows             :", len(whatsapp))
print("Unique messages  :", whatsapp.message_id.nunique())
print("Unique customers :", whatsapp.customer_id.nunique())
print("Period           :", whatsapp.sent_at.min(), "→", whatsapp.sent_at.max())

display(whatsapp.head())

Rows             : 75406
Unique messages  : 75406
Unique customers : 11724
Period           : 2026-06-01 09:18:00 → 2026-08-31 20:59:00


,message_id,customer_id,sent_at,template,n_msgs_last_14d,days_past_due,outstanding_balance_brl,monthly_salary_brl,payday_day_of_month,n_prior_transactions,account_age_months,days_since_last_app_login,state_uf,delivery_status,interaction,paid_within_72h,amount_paid_brl
0,M0000001,C001934,2026-06-01 09:18:00,pix_link,0,1,867.63,3000.0,5,2,4,44,PB,delivered,none,0,0.0
1,M0000002,C007836,2026-06-01 09:18:00,friendly_reminder,0,1,790.21,3260.0,30,3,3,9,RJ,delivered,read,0,0.0
2,M0000003,C004887,2026-06-01 09:48:00,friendly_reminder,0,1,1030.00,3120.0,5,2,6,5,PA,delivered,none,0,0.0
3,M0000004,C009207,2026-06-01 09:53:00,friendly_reminder,0,1,250.94,1200.0,30,11,11,26,SP,delivered,read,0,0.0
4,M0000005,C003478,2026-06-01 09:54:00,friendly_reminder,0,1,511.52,1740.0,10,2,1,23,SP,delivered,read,0,0.0


In [4]:
expected_domains = {
    "template": {"friendly_reminder","urgent_reminder","discount_offer","pix_link"},
    "delivery_status": {"delivered","failed_invalid_number","failed_blocked","failed_unreachable"},
    "interaction": {"none","read","replied","clicked_link"},
    "paid_within_72h": {0,1},
    "payday_day_of_month": {1,5,10,15,20,25,30},
}

domain_audit = []
for col, expected in expected_domains.items():
    observed = set(whatsapp[col].dropna().unique())
    domain_audit.append({
        "column": col,
        "observed_n": len(observed),
        "unexpected": sorted(observed - expected),
        "compliant": observed.issubset(expected),
    })
display(pd.DataFrame(domain_audit))

assert (whatsapp.loc[whatsapp.delivery_status != "delivered", "interaction"] == "none").all()
assert (whatsapp["outstanding_balance_brl"] >= 0).all()
assert (whatsapp["amount_paid_brl"] >= 0).all()
assert (whatsapp["days_past_due"] >= 1).all()

,column,observed_n,unexpected,compliant
0,template,4,[],True
1,delivery_status,4,[],True
2,interaction,4,[],True
3,paid_within_72h,2,[],True
4,payday_day_of_month,7,[],True


## 4.2 Grain and within-customer consistency

In [5]:
static_cols = [
    "monthly_salary_brl","payday_day_of_month","n_prior_transactions",
    "account_age_months","state_uf"
]

consistency = []
for col in static_cols:
    nunique = whatsapp.groupby("customer_id")[col].nunique(dropna=False)
    consistency.append({
        "column": col,
        "customers_with_multiple_values": int((nunique > 1).sum()),
        "max_values_per_customer": int(nunique.max()),
    })
display(pd.DataFrame(consistency))

,column,customers_with_multiple_values,max_values_per_customer
0,monthly_salary_brl,0,1
1,payday_day_of_month,0,1
2,n_prior_transactions,0,1
3,account_age_months,0,1
4,state_uf,0,1


## 4.3 Interaction-level temporal enrichment

In [6]:
x = whatsapp.sort_values(["customer_id","sent_at","message_id"]).copy()
g = x.groupby("customer_id", sort=False)

x["attempt_number"] = g.cumcount() + 1
x["n_attempts_customer"] = g["message_id"].transform("size")
x["attempt_share_of_journey"] = x["attempt_number"] / x["n_attempts_customer"]
x["journey_half"] = np.where(x["attempt_share_of_journey"] <= .5, "early", "late")

# Infer collections entry date: DPD=1 is first collections day.
x["collections_start_date_inferred"] = (
    x["sent_at"].dt.normalize() - pd.to_timedelta(x["days_past_due"] - 1, unit="D")
)

# Temporal gaps / previous states
x["prev_sent_at"] = g["sent_at"].shift()
x["gap_hours_from_prev"] = (x["sent_at"] - x["prev_sent_at"]).dt.total_seconds() / 3600
x["gap_days_from_prev"] = x["gap_hours_from_prev"] / 24

for col in ["template","interaction","delivery_status","days_past_due","outstanding_balance_brl"]:
    x[f"prev_{col}"] = g[col].shift()

x["template_changed"] = (x["template"] != x["prev_template"]) & x["prev_template"].notna()
x["interaction_changed"] = (x["interaction"] != x["prev_interaction"]) & x["prev_interaction"].notna()
x["delivery_changed"] = (x["delivery_status"] != x["prev_delivery_status"]) & x["prev_delivery_status"].notna()

# DPD buckets
dpd_bins = [0,7,15,30,60,np.inf]
dpd_labels = ["01_07","08_15","16_30","31_60","61_plus"]
x["dpd_bucket"] = pd.cut(x["days_past_due"], bins=dpd_bins, labels=dpd_labels)
x["prev_dpd_bucket"] = g["dpd_bucket"].shift()
x["dpd_bucket_changed"] = (x["dpd_bucket"].astype(str) != x["prev_dpd_bucket"].astype(str)) & x["prev_dpd_bucket"].notna()

# Send timing
x["send_date"] = x["sent_at"].dt.date
x["send_hour"] = x["sent_at"].dt.hour
x["send_day_of_week"] = x["sent_at"].dt.day_name()
x["send_day_of_month"] = x["sent_at"].dt.day
x["send_month"] = x["sent_at"].dt.month
x["is_weekend"] = x["sent_at"].dt.dayofweek >= 5

x["daypart"] = pd.cut(
    x["send_hour"],
    bins=[8,11,13,17,20],
    labels=["morning_09_11","lunch_12_13","afternoon_14_17","evening_18_20"],
    include_lowest=True
)
x["month_period"] = pd.cut(
    x["send_day_of_month"],
    bins=[0,10,20,31],
    labels=["beginning_01_10","middle_11_20","end_21_31"]
)

# Salary / exposure / app-recency segmentation
x["salary_bucket"] = pd.cut(
    x["monthly_salary_brl"],
    bins=[-np.inf,1500,2500,4000,6000,np.inf],
    labels=["le_1500","1501_2500","2501_4000","4001_6000","gt_6000"]
)
x["balance_bucket"] = pd.cut(
    x["outstanding_balance_brl"],
    bins=[-np.inf,500,1000,1500,2000,np.inf],
    labels=["le_500","501_1000","1001_1500","1501_2000","gt_2000"]
)
x["app_recency_bucket"] = pd.cut(
    x["days_since_last_app_login"],
    bins=[-np.inf,3,7,14,30,60,np.inf],
    labels=["00_03","04_07","08_14","15_30","31_60","61_plus"]
)
x["balance_to_salary_ratio"] = np.where(
    x["monthly_salary_brl"] > 0,
    x["outstanding_balance_brl"] / x["monthly_salary_brl"],
    np.nan
)

# Payday distance using circular distance within each calendar month.
def payday_features(row):
    sent = pd.Timestamp(row["sent_at"]).normalize()
    target_day = int(row["payday_day_of_month"])
    month_end = sent.days_in_month
    target_day = min(target_day, month_end)
    payday = sent.replace(day=target_day)
    delta = (payday - sent).days
    return pd.Series([delta, abs(delta)])

x[["days_to_payday_same_month","abs_days_to_payday_same_month"]] = x.apply(payday_features, axis=1)
x["send_on_payday"] = x["days_to_payday_same_month"].eq(0)
x["send_within_2d_payday"] = x["abs_days_to_payday_same_month"].le(2)
x["send_within_3d_payday"] = x["abs_days_to_payday_same_month"].le(3)
x["send_within_5d_payday"] = x["abs_days_to_payday_same_month"].le(5)
x["send_before_payday"] = x["days_to_payday_same_month"].gt(0)
x["send_after_payday"] = x["days_to_payday_same_month"].lt(0)

# Send-level operational flags
x["delivered_flag"] = x["delivery_status"].eq("delivered").astype(int)
x["failed_flag"] = (~x["delivery_status"].eq("delivered")).astype(int)
x["read_flag"] = x["interaction"].eq("read").astype(int)
x["reply_flag"] = x["interaction"].eq("replied").astype(int)
x["click_flag"] = x["interaction"].eq("clicked_link").astype(int)
x["engaged_flag"] = x["interaction"].isin(["read","replied","clicked_link"]).astype(int)
x["payment_flag"] = (x["amount_paid_brl"].fillna(0) > 0).astype(int)

# Previous-known bad contactability / potentially wasteful sends.
x["prior_blocked"] = g["delivery_status"].transform(
    lambda s: s.eq("failed_blocked").shift(fill_value=False).cummax()
)
x["prior_invalid"] = g["delivery_status"].transform(
    lambda s: s.eq("failed_invalid_number").shift(fill_value=False).cummax()
)
x["prior_failure_count"] = g["failed_flag"].cumsum() - x["failed_flag"]
x["send_after_blocked_flag"] = x["prior_blocked"].astype(int)
x["send_after_invalid_flag"] = x["prior_invalid"].astype(int)
x["send_after_repeated_failure_flag"] = (x["prior_failure_count"] >= 2).astype(int)

# Combination labels retained for pivot features later.
x["template_x_interaction"] = x["template"].astype(str) + "__" + x["interaction"].astype(str)
x["template_x_delivery"] = x["template"].astype(str) + "__" + x["delivery_status"].astype(str)
x["template_x_daypart"] = x["template"].astype(str) + "__" + x["daypart"].astype(str)

display(x.head())

,message_id,customer_id,sent_at,template,n_msgs_last_14d,days_past_due,outstanding_balance_brl,monthly_salary_brl,payday_day_of_month,n_prior_transactions,account_age_months,days_since_last_app_login,state_uf,delivery_status,interaction,paid_within_72h,amount_paid_brl,attempt_number,n_attempts_customer,attempt_share_of_journey,journey_half,collections_start_date_inferred,prev_sent_at,gap_hours_from_prev,gap_days_from_prev,prev_template,prev_interaction,prev_delivery_status,prev_days_past_due,prev_outstanding_balance_brl,template_changed,interaction_changed,delivery_changed,dpd_bucket,prev_dpd_bucket,dpd_bucket_changed,send_date,send_hour,send_day_of_week,send_day_of_month,send_month,is_weekend,daypart,month_period,salary_bucket,balance_bucket,app_recency_bucket,balance_to_salary_ratio,days_to_payday_same_month,abs_days_to_payday_same_month,send_on_payday,send_within_2d_payday,send_within_3d_payday,send_within_5d_payday,send_before_payday,send_after_payday,delivered_flag,failed_flag,read_flag,reply_flag,click_flag,engaged_flag,payment_flag,prior_blocked,prior_invalid,prior_failure_count,send_after_blocked_flag,send_after_invalid_flag,send_after_repeated_failure_flag,template_x_interaction,template_x_delivery,template_x_daypart
18747,M0018748,C000001,2026-07-06 13:29:00,pix_link,0,3,934.58,1680.0,5,5,2,38,ES,delivered,none,0,0.0,1,7,0.142857,early,2026-07-04,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,False,False,01_07,NaN,False,2026-07-06,13,Monday,6,7,False,lunch_12_13,beginning_01_10,1501_2500,501_1000,31_60,0.556298,-1,1,False,True,True,True,False,True,1,0,0,0,0,0,0,False,False,0,0,0,0,pix_link__none,pix_link__delivered,pix_link__lunch_12_13
19915,M0019916,C000001,2026-07-07 14:48:00,friendly_reminder,1,4,934.58,1680.0,5,5,2,39,ES,delivered,clicked_link,0,0.0,2,7,0.285714,early,2026-07-04,2026-07-06 13:29:00,25.316667,1.054861,pix_link,none,delivered,3.0,934.58,True,True,False,01_07,01_07,False,2026-07-07,14,Tuesday,7,7,False,afternoon_14_17,beginning_01_10,1501_2500,501_1000,31_60,0.556298,-2,2,False,True,True,True,False,True,1,0,0,0,1,1,0,False,False,0,0,0,0,friendly_reminder__clicked_link,friendly_reminder__delivered,friendly_reminder__afternoon_14_17
20535,M0020536,C000001,2026-07-08 10:46:00,pix_link,2,5,934.58,1680.0,5,5,2,1,ES,delivered,none,0,0.0,3,7,0.428571,early,2026-07-04,2026-07-07 14:48:00,19.966667,0.831944,friendly_reminder,clicked_link,delivered,4.0,934.58,True,True,False,01_07,01_07,False,2026-07-08,10,Wednesday,8,7,False,morning_09_11,beginning_01_10,1501_2500,501_1000,00_03,0.556298,-3,3,False,False,True,True,False,True,1,0,0,0,0,0,0,False,False,0,0,0,0,pix_link__none,pix_link__delivered,pix_link__morning_09_11
23753,M0023754,C000001,2026-07-11 13:11:00,urgent_reminder,3,8,934.58,1680.0,5,5,2,4,ES,delivered,none,0,0.0,4,7,0.571429,late,2026-07-04,2026-07-08 10:46:00,74.416667,3.100694,pix_link,none,delivered,5.0,934.58,True,False,False,08_15,01_07,True,2026-07-11,13,Saturday,11,7,True,lunch_12_13,middle_11_20,1501_2500,501_1000,04_07,0.556298,-6,6,False,False,False,False,False,True,1,0,0,0,0,0,0,False,False,0,0,0,0,urgent_reminder__none,urgent_reminder__delivered,urgent_reminder__lunch_12_13
27129,M0027130,C000001,2026-07-15 12:54:00,urgent_reminder,4,12,934.58,1680.0,5,5,2,8,ES,delivered,none,0,0.0,5,7,0.714286,late,2026-07-04,2026-07-11 13:11:00,95.716667,3.988194,urgent_reminder,none,delivered,8.0,934.58,False,False,False,08_15,08_15,False,2026-07-15,12,Wednesday,15,7,False,lunch_12_13,middle_11_20,1501_2500,501_1000,08_14,0.556298,-10,10,False,False,False,False,False,True,1,0,0,0,0,0,0,False,False,0,0,0,0,urgent_reminder__none,urgent_reminder__delivered,urgent_reminder__lunch_12_13


## 4.4 Payment attribution and balance-trajectory audit

Um pagamento pode ser associado a uma janela de 72 horas após cada envio.  
Como um mesmo cliente pode receber múltiplas mensagens em um curto intervalo, essas janelas de atribuição podem se sobrepor. Portanto:  

- sum(amount_paid_brl) é tratado como total_attributed_payment_72h (pagamento total atribuído às janelas de 72h), e não automaticamente como recuperação econômica total;
- a redução do saldo devedor é reconstruída de forma independente a partir da trajetória dos saldos observados (outstanding_balance_brl);
- divergências entre pagamento atribuído e redução observada do saldo são identificadas e sinalizadas para investigação e interpretação nas etapas analíticas posteriores.

In [7]:
# Basic within-customer monotonicity diagnostics
trajectory_audit = g.agg(
    initial_balance=("outstanding_balance_brl","first"),
    last_balance=("outstanding_balance_brl","last"),
    min_balance=("outstanding_balance_brl","min"),
    max_balance=("outstanding_balance_brl","max"),
    initial_dpd=("days_past_due","first"),
    last_dpd=("days_past_due","last"),
    max_dpd=("days_past_due","max"),
    total_attributed_payment_72h=("amount_paid_brl","sum"),
).reset_index()

trajectory_audit["observed_balance_reduction"] = (
    trajectory_audit["initial_balance"] - trajectory_audit["last_balance"]
)
trajectory_audit["attributed_payment_over_initial_flag"] = (
    trajectory_audit["total_attributed_payment_72h"] > trajectory_audit["initial_balance"] + 1e-9
)

balance_increase = g["outstanding_balance_brl"].apply(lambda s: s.diff().gt(1e-9).any())
dpd_decrease = g["days_past_due"].apply(lambda s: s.diff().lt(0).any())

print("Customers with balance increase:", int(balance_increase.sum()))
print("Customers with DPD decrease     :", int(dpd_decrease.sum()))
print("Attributed payment > initial balance:", int(trajectory_audit.attributed_payment_over_initial_flag.sum()))

display(trajectory_audit.describe(include="all").T)

Customers with balance increase: 0
Customers with DPD decrease     : 0
Attributed payment > initial balance: 5


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,11724,11724,C000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
initial_balance,11724.0,NaN,NaN,NaN,850.011969,485.334446,250.0,464.31,751.185,1138.19,2000.0
last_balance,11724.0,NaN,NaN,NaN,793.288091,491.551141,51.61,397.175,684.32,1075.505,2000.0
min_balance,11724.0,NaN,NaN,NaN,793.288091,491.551141,51.61,397.175,684.32,1075.505,2000.0
max_balance,11724.0,NaN,NaN,NaN,850.011969,485.334446,250.0,464.31,751.185,1138.19,2000.0
initial_dpd,11724.0,NaN,NaN,NaN,3.056551,2.480247,1.0,1.0,2.0,4.0,32.0
last_dpd,11724.0,NaN,NaN,NaN,27.463238,18.935834,1.0,10.0,25.0,45.0,60.0
max_dpd,11724.0,NaN,NaN,NaN,27.463238,18.935834,1.0,10.0,25.0,45.0,60.0
total_attributed_payment_72h,11724.0,NaN,NaN,NaN,295.061865,462.945398,0.0,0.0,0.0,478.1325,2000.0
observed_balance_reduction,11724.0,NaN,NaN,NaN,56.723878,180.471099,0.0,0.0,0.0,0.0,1753.28


## 4.4 — Payment × Balance reconciliation

In [8]:
# ============================================================
# Payment × Balance reconciliation
# ============================================================

import numpy as np
import pandas as pd

# Make sure observations are in chronological order
whatsapp_recon = whatsapp.sort_values(
    ["customer_id", "sent_at"]
).copy()

g_recon = whatsapp_recon.groupby(
    "customer_id",
    sort=False
)

# ------------------------------------------------------------
# 1. Customer-level reconciliation
# ------------------------------------------------------------

payment_balance_recon = g_recon.agg(
    n_interactions=("customer_id", "size"),

    initial_balance=("outstanding_balance_brl", "first"),
    last_balance=("outstanding_balance_brl", "last"),
    min_balance=("outstanding_balance_brl", "min"),
    max_balance=("outstanding_balance_brl", "max"),

    initial_dpd=("days_past_due", "first"),
    last_dpd=("days_past_due", "last"),
    max_dpd=("days_past_due", "max"),

    total_attributed_payment_72h=("amount_paid_brl", "sum"),

    first_interaction_at=("sent_at", "first"),
    last_interaction_at=("sent_at", "last"),
).reset_index()


# ------------------------------------------------------------
# 2. Reconstruct observed economic movement
# ------------------------------------------------------------

payment_balance_recon["observed_balance_reduction"] = (
    payment_balance_recon["initial_balance"]
    - payment_balance_recon["last_balance"]
)

payment_balance_recon["observed_recovery_rate"] = np.where(
    payment_balance_recon["initial_balance"] > 0,
    payment_balance_recon["observed_balance_reduction"]
    / payment_balance_recon["initial_balance"],
    np.nan
)

payment_balance_recon["attributed_payment_rate"] = np.where(
    payment_balance_recon["initial_balance"] > 0,
    payment_balance_recon["total_attributed_payment_72h"]
    / payment_balance_recon["initial_balance"],
    np.nan
)


# ------------------------------------------------------------
# 3. Basic payment / balance flags
# ------------------------------------------------------------

payment_balance_recon["has_attributed_payment"] = (
    payment_balance_recon["total_attributed_payment_72h"] > 1e-9
)

payment_balance_recon["has_observed_balance_reduction"] = (
    payment_balance_recon["observed_balance_reduction"] > 1e-9
)

payment_balance_recon["payment_over_initial_balance"] = (
    payment_balance_recon["total_attributed_payment_72h"]
    > payment_balance_recon["initial_balance"] + 1e-9
)


# ------------------------------------------------------------
# 4. Payment × observed balance movement matrix
# ------------------------------------------------------------

payment_balance_recon["reconciliation_group"] = np.select(
    [
        (
            ~payment_balance_recon["has_attributed_payment"]
            & ~payment_balance_recon["has_observed_balance_reduction"]
        ),
        (
            payment_balance_recon["has_attributed_payment"]
            & payment_balance_recon["has_observed_balance_reduction"]
        ),
        (
            payment_balance_recon["has_attributed_payment"]
            & ~payment_balance_recon["has_observed_balance_reduction"]
        ),
        (
            ~payment_balance_recon["has_attributed_payment"]
            & payment_balance_recon["has_observed_balance_reduction"]
        ),
    ],
    [
        "no_payment_no_balance_reduction",
        "payment_and_balance_reduction",
        "payment_without_balance_reduction",
        "balance_reduction_without_payment",
    ],
    default="other",
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

reconciliation_summary = (
    payment_balance_recon
    .groupby("reconciliation_group", dropna=False)
    .agg(
        customers=("customer_id", "nunique"),
        initial_balance=("initial_balance", "sum"),
        observed_balance_reduction=("observed_balance_reduction", "sum"),
        attributed_payment_72h=("total_attributed_payment_72h", "sum"),
    )
    .reset_index()
)

reconciliation_summary["customer_pct"] = (
    reconciliation_summary["customers"]
    / reconciliation_summary["customers"].sum()
)

reconciliation_summary["observed_recovery_rate"] = np.where(
    reconciliation_summary["initial_balance"] > 0,
    reconciliation_summary["observed_balance_reduction"]
    / reconciliation_summary["initial_balance"],
    np.nan
)

display(
    reconciliation_summary.sort_values(
        "customers",
        ascending=False
    )
)

,reconciliation_group,customers,initial_balance,observed_balance_reduction,attributed_payment_72h,customer_pct,observed_recovery_rate
0,no_payment_no_balance_reduction,6831,5984825.45,0.00,0.00,0.582651,0.000000
2,payment_without_balance_reduction,3357,2737370.75,0.00,2616502.16,0.286336,0.000000
1,payment_and_balance_reduction,1536,1243344.13,665030.75,842803.14,0.131013,0.534873


| Grupo                               | O que significa                                   |
| ----------------------------------- | ------------------------------------------------- |
| `no_payment_no_balance_reduction`   | nenhuma evidência de pagamento nem redução        |
| `payment_and_balance_reduction`     | os dois sinais aparecem                           |
| `payment_without_balance_reduction` | pagamento atribuído, mas saldo observado não caiu |
| `balance_reduction_without_payment` | saldo caiu sem pagamento atribuído às mensagens   |


## 4.5 Canonical customer-level base

A camada canônica retém deliberadamente tanto o estado quanto a jornada:

- valores primeiro / último / mínimo / máximo para medidas que variam ao longo do tempo;
- contagens e taxas;
- combinações de tratamento × resposta;
- sequências;
- contagens de transições;
- tempo até o primeiro evento;
- sinais do início versus final da jornada;
- proxies de desperdício operacional.


In [9]:
def safe_rate(num, den):
    return np.where(den > 0, num / den, np.nan)

def first_value(s):
    return s.iloc[0] if len(s) else np.nan

def last_value(s):
    return s.iloc[-1] if len(s) else np.nan

def seq_json(s):
    vals = [None if pd.isna(v) else str(v) for v in s]
    return json.dumps(vals, ensure_ascii=False)

base = g.agg(
    # Profile
    monthly_salary_brl=("monthly_salary_brl","first"),
    payday_day_of_month=("payday_day_of_month","first"),
    n_prior_transactions=("n_prior_transactions","first"),
    account_age_months=("account_age_months","first"),
    state_uf=("state_uf","first"),

    # Journey timing
    first_send_at=("sent_at","first"),
    last_send_at=("sent_at","last"),
    n_messages=("message_id","size"),
    n_active_contact_days=("send_date","nunique"),

    # Delinquency
    collections_start_date_inferred=("collections_start_date_inferred","first"),
    initial_dpd=("days_past_due","first"),
    last_dpd=("days_past_due","last"),
    max_dpd=("days_past_due","max"),
    min_dpd=("days_past_due","min"),
    initial_dpd_bucket=("dpd_bucket","first"),
    last_dpd_bucket=("dpd_bucket","last"),
    n_dpd_buckets_observed=("dpd_bucket","nunique"),
    n_dpd_bucket_transitions=("dpd_bucket_changed","sum"),

    # Exposure
    initial_balance_brl=("outstanding_balance_brl","first"),
    last_balance_brl=("outstanding_balance_brl","last"),
    min_balance_brl=("outstanding_balance_brl","min"),
    max_balance_brl=("outstanding_balance_brl","max"),

    # App recency
    initial_days_since_last_app_login=("days_since_last_app_login","first"),
    last_days_since_last_app_login=("days_since_last_app_login","last"),
    min_days_since_last_app_login=("days_since_last_app_login","min"),
    max_days_since_last_app_login=("days_since_last_app_login","max"),

    # Pressure
    avg_n_msgs_last_14d=("n_msgs_last_14d","mean"),
    max_n_msgs_last_14d=("n_msgs_last_14d","max"),
    last_n_msgs_last_14d=("n_msgs_last_14d","last"),
    avg_gap_hours=("gap_hours_from_prev","mean"),
    median_gap_hours=("gap_hours_from_prev","median"),
    min_gap_hours=("gap_hours_from_prev","min"),
    max_gap_hours=("gap_hours_from_prev","max"),

    # Contactability
    n_delivered=("delivered_flag","sum"),
    n_failed=("failed_flag","sum"),
    n_failed_invalid_number=("delivery_status", lambda s: s.eq("failed_invalid_number").sum()),
    n_failed_blocked=("delivery_status", lambda s: s.eq("failed_blocked").sum()),
    n_failed_unreachable=("delivery_status", lambda s: s.eq("failed_unreachable").sum()),
    first_delivery_status=("delivery_status","first"),
    last_delivery_status=("delivery_status","last"),

    # Engagement
    n_none=("interaction", lambda s: s.eq("none").sum()),
    n_reads=("read_flag","sum"),
    n_replies=("reply_flag","sum"),
    n_clicks=("click_flag","sum"),
    n_engaged=("engaged_flag","sum"),
    first_interaction=("interaction","first"),
    last_interaction=("interaction","last"),

    # Treatment
    first_template=("template","first"),
    last_template=("template","last"),
    n_distinct_templates=("template","nunique"),
    n_template_switches=("template_changed","sum"),

    # Payment association
    any_payment_72h=("payment_flag","max"),
    n_payment_associated_sends=("payment_flag","sum"),
    total_attributed_payment_72h_brl=("amount_paid_brl","sum"),

    # Timing
    avg_send_hour=("send_hour","mean"),
    first_send_hour=("send_hour","first"),
    last_send_hour=("send_hour","last"),
    n_weekend_sends=("is_weekend","sum"),
    n_payday_sends=("send_on_payday","sum"),
    n_sends_within_2d_payday=("send_within_2d_payday","sum"),
    n_sends_within_3d_payday=("send_within_3d_payday","sum"),
    n_sends_within_5d_payday=("send_within_5d_payday","sum"),

    # Waste proxies
    n_sends_after_blocked=("send_after_blocked_flag","sum"),
    n_sends_after_invalid=("send_after_invalid_flag","sum"),
    n_sends_after_repeated_failure=("send_after_repeated_failure_flag","sum"),
).reset_index()

base["observed_journey_days"] = (
    base["last_send_at"].dt.normalize() - base["first_send_at"].dt.normalize()
).dt.days + 1
base["dpd_span"] = base["last_dpd"] - base["initial_dpd"]
base["crossed_dpd_bucket_flag"] = (base["n_dpd_buckets_observed"] > 1).astype(int)

base["observed_balance_reduction_brl"] = base["initial_balance_brl"] - base["last_balance_brl"]
base["observed_balance_reduction_pct"] = safe_rate(
    base["observed_balance_reduction_brl"], base["initial_balance_brl"]
)
base["any_balance_reduction"] = (base["observed_balance_reduction_brl"] > 1e-9).astype(int)
base["remaining_balance_flag"] = (base["last_balance_brl"] > 1e-9).astype(int)
base["remaining_balance_pct"] = safe_rate(base["last_balance_brl"], base["initial_balance_brl"])
base["partial_reduction_still_open_flag"] = (
    (base["observed_balance_reduction_brl"] > 1e-9) & (base["last_balance_brl"] > 1e-9)
).astype(int)

base["initial_balance_to_salary"] = safe_rate(base["initial_balance_brl"], base["monthly_salary_brl"])
base["last_balance_to_salary"] = safe_rate(base["last_balance_brl"], base["monthly_salary_brl"])

base["delivery_rate"] = safe_rate(base["n_delivered"], base["n_messages"])
base["failure_rate"] = safe_rate(base["n_failed"], base["n_messages"])
base["read_rate_all_sends"] = safe_rate(base["n_reads"], base["n_messages"])
base["reply_rate_all_sends"] = safe_rate(base["n_replies"], base["n_messages"])
base["click_rate_all_sends"] = safe_rate(base["n_clicks"], base["n_messages"])
base["engagement_rate_all_sends"] = safe_rate(base["n_engaged"], base["n_messages"])
base["engagement_rate_delivered"] = safe_rate(base["n_engaged"], base["n_delivered"])

base["messages_per_active_day"] = safe_rate(base["n_messages"], base["n_active_contact_days"])
base["weekend_send_share"] = safe_rate(base["n_weekend_sends"], base["n_messages"])
base["payday_2d_send_share"] = safe_rate(base["n_sends_within_2d_payday"], base["n_messages"])
base["payday_3d_send_share"] = safe_rate(base["n_sends_within_3d_payday"], base["n_messages"])
base["payday_5d_send_share"] = safe_rate(base["n_sends_within_5d_payday"], base["n_messages"])

base["message_cost_brl"] = base["n_messages"] * 1.0
base["failed_message_cost_brl"] = base["n_failed"] * 1.0
base["wasted_send_rate"] = base["failure_rate"]
base["attributed_recovery_per_send_brl"] = safe_rate(
    base["total_attributed_payment_72h_brl"], base["n_messages"]
)
base["attributed_net_recovery_after_message_cost_brl"] = (
    base["total_attributed_payment_72h_brl"] - base["message_cost_brl"]
)
base["attributed_payment_to_initial_balance"] = safe_rate(
    base["total_attributed_payment_72h_brl"], base["initial_balance_brl"]
)
base["attributed_payment_over_initial_flag"] = (
    base["total_attributed_payment_72h_brl"] > base["initial_balance_brl"] + 1e-9
).astype(int)

base.head()

,customer_id,monthly_salary_brl,payday_day_of_month,n_prior_transactions,account_age_months,state_uf,first_send_at,last_send_at,n_messages,n_active_contact_days,collections_start_date_inferred,initial_dpd,last_dpd,max_dpd,min_dpd,initial_dpd_bucket,last_dpd_bucket,n_dpd_buckets_observed,n_dpd_bucket_transitions,initial_balance_brl,last_balance_brl,min_balance_brl,max_balance_brl,initial_days_since_last_app_login,last_days_since_last_app_login,min_days_since_last_app_login,max_days_since_last_app_login,avg_n_msgs_last_14d,max_n_msgs_last_14d,last_n_msgs_last_14d,avg_gap_hours,median_gap_hours,min_gap_hours,max_gap_hours,n_delivered,n_failed,n_failed_invalid_number,n_failed_blocked,n_failed_unreachable,first_delivery_status,last_delivery_status,n_none,n_reads,n_replies,n_clicks,n_engaged,first_interaction,last_interaction,first_template,last_template,n_distinct_templates,n_template_switches,any_payment_72h,n_payment_associated_sends,total_attributed_payment_72h_brl,avg_send_hour,first_send_hour,last_send_hour,n_weekend_sends,n_payday_sends,n_sends_within_2d_payday,n_sends_within_3d_payday,n_sends_within_5d_payday,n_sends_after_blocked,n_sends_after_invalid,n_sends_after_repeated_failure,observed_journey_days,dpd_span,crossed_dpd_bucket_flag,observed_balance_reduction_brl,observed_balance_reduction_pct,any_balance_reduction,remaining_balance_flag,remaining_balance_pct,partial_reduction_still_open_flag,initial_balance_to_salary,last_balance_to_salary,delivery_rate,failure_rate,read_rate_all_sends,reply_rate_all_sends,click_rate_all_sends,engagement_rate_all_sends,engagement_rate_delivered,messages_per_active_day,weekend_send_share,payday_2d_send_share,payday_3d_send_share,payday_5d_send_share,message_cost_brl,failed_message_cost_brl,wasted_send_rate,attributed_recovery_per_send_brl,attributed_net_recovery_after_message_cost_brl,attributed_payment_to_initial_balance,attributed_payment_over_initial_flag
0,C000001,1680.0,5,5,2,ES,2026-07-06 13:29:00,2026-07-29 09:14:00,7,7,2026-07-04,3,26,26,3,01_07,16_30,3,2,934.58,934.58,934.58,934.58,38,22,1,39,2.428571,5,2,91.291667,61.208333,19.966667,284.333333,7,0,0,0,0,delivered,delivered,6,0,0,1,1,none,none,pix_link,urgent_reminder,3,3,1,1,934.58,11.857143,13,9,1,0,2,3,3,0,0,0,24,23,1,0.0,0.0,0,1,1.0,0,0.556298,0.556298,1.0,0.0,0.000000,0.000000,0.142857,0.142857,0.142857,1.0,0.142857,0.285714,0.428571,0.428571,7.0,0.0,0.0,133.511429,927.58,1.0,0
1,C000002,2670.0,20,4,7,BA,2026-07-29 10:31:00,2026-08-10 14:53:00,4,4,2026-07-26,4,16,16,4,01_07,16_30,3,2,1143.79,1143.79,1143.79,1143.79,15,27,15,27,1.500000,3,3,97.455556,73.383333,23.000000,195.983333,4,0,0,0,0,delivered,delivered,2,2,0,0,2,read,none,pix_link,pix_link,3,3,0,0,0.00,10.750000,10,14,1,0,0,0,0,0,0,0,13,12,1,0.0,0.0,0,1,1.0,0,0.428386,0.428386,1.0,0.0,0.500000,0.000000,0.000000,0.500000,0.500000,1.0,0.250000,0.000000,0.000000,0.000000,4.0,0.0,0.0,0.000000,-4.00,0.0,0
2,C000003,1200.0,25,2,1,SP,2026-06-09 09:52:00,2026-08-05 10:54:00,12,12,2026-06-07,3,60,60,3,01_07,31_60,4,3,758.22,758.22,758.22,758.22,13,36,13,36,3.000000,7,1,124.457576,51.750000,17.733333,750.516667,12,0,0,0,0,delivered,delivered,8,2,1,1,4,none,replied,friendly_reminder,pix_link,3,6,0,0,0.00,12.833333,9,10,3,0,0,1,4,0,0,0,58,57,1,0.0,0.0,0,1,1.0,0,0.631850,0.631850,1.0,0.0,0.166667,0.083333,0.083333,0.333333,0.333333,1.0,0.250000,0.000000,0.083333,0.333333,12.0,0.0,0.0,0.000000,-12.00,0.0,0
3,C000004,4500.0,5,2,1,GO,2026-06-30 17:46:00,2026-07-10 10:57:00,5,5,2026-06-30,1,11,11,1,01_07,08_15,2,1,1331.09,1331.09,1331.09,1331.09,1,11,1,11,2.000000,4,4,58.295833,58.950000,17.466667,97.816667,5,0,0,0,0,delivered,delivered,1,4,0,0,4,read,none,friendly_reminder,urgent_reminder,2,1,1,1,1331.09,13.600000,17,10,0,0,1,2,4,0,0,0,11,10,1,0.0,0.0,0,1,1.0,0,0.295798,0.295798,1.0,0.0,0.800000,0.000000,0.000000,0.800000,0.800000,1.0,0.000000,0.200000,0.400000,0.800000,5.0,0.0,0.0,266.218000,1326.09,1.0,0
4,C000005,3200.0,5,1,5,RS,2026-08-05 14:36:00,2026-08-25 19:13:00,5,5,2026-0

### Categorical openings

Counts and shares are generated for treatment, interaction, delivery, daypart, weekday, month-period, DPD stage, app-recency bucket, salary bucket and key cross-features such as `pix_link × read`.

In [10]:
def add_count_share_pivot(canonical, source, category_col, prefix):
    counts = pd.crosstab(source["customer_id"], source[category_col])
    counts.columns = [f"{prefix}__{str(c)}__n" for c in counts.columns]
    counts = counts.reset_index()

    out = canonical.merge(counts, on="customer_id", how="left")
    new_count_cols = [c for c in out.columns if c.startswith(prefix + "__") and c.endswith("__n")]
    out[new_count_cols] = out[new_count_cols].fillna(0).astype(int)

    for c in new_count_cols:
        share_col = c[:-3] + "__share"
        out[share_col] = safe_rate(out[c], out["n_messages"])
    return out

canonical = base.copy()

for category_col, prefix in [
    ("template","template"),
    ("interaction","interaction"),
    ("delivery_status","delivery"),
    ("daypart","daypart"),
    ("send_day_of_week","weekday"),
    ("month_period","month_period"),
    ("dpd_bucket","dpd_bucket"),
    ("app_recency_bucket","app_recency"),
    ("salary_bucket","salary_bucket"),
    ("template_x_interaction","template_x_interaction"),
    ("template_x_delivery","template_x_delivery"),
    ("template_x_daypart","template_x_daypart"),
]:
    canonical = add_count_share_pivot(canonical, x, category_col, prefix)

print("Canonical shape after categorical openings:", canonical.shape)

Canonical shape after categorical openings: (11724, 274)


### Journey sequences and first-event features

Sequences are retained explicitly rather than discarded by aggregation. They support later journey analysis without pretending that historical policy exposure is a customer attribute.

In [11]:
sequence_features = g.agg(
    template_sequence=("template", seq_json),
    interaction_sequence=("interaction", seq_json),
    delivery_sequence=("delivery_status", seq_json),
    daypart_sequence=("daypart", seq_json),
    weekday_sequence=("send_day_of_week", seq_json),
    dpd_bucket_sequence=("dpd_bucket", seq_json),
    send_hour_sequence=("send_hour", seq_json),
).reset_index()

def first_event_table(flag_col, prefix):
    z = x.loc[x[flag_col].eq(1)].copy()
    if z.empty:
        return pd.DataFrame({"customer_id": x.customer_id.unique()})
    z = z.sort_values(["customer_id","sent_at","message_id"]).groupby("customer_id").first().reset_index()
    return z[[
        "customer_id","attempt_number","sent_at","days_past_due","template",
        "interaction","delivery_status","send_hour","daypart","send_day_of_week",
        "n_msgs_last_14d","outstanding_balance_brl"
    ]].rename(columns={
        "attempt_number": f"messages_until_first_{prefix}",
        "sent_at": f"first_{prefix}_send_at",
        "days_past_due": f"dpd_at_first_{prefix}",
        "template": f"template_at_first_{prefix}",
        "interaction": f"interaction_at_first_{prefix}",
        "delivery_status": f"delivery_at_first_{prefix}",
        "send_hour": f"hour_at_first_{prefix}",
        "daypart": f"daypart_at_first_{prefix}",
        "send_day_of_week": f"weekday_at_first_{prefix}",
        "n_msgs_last_14d": f"pressure14d_at_first_{prefix}",
        "outstanding_balance_brl": f"balance_at_first_{prefix}",
    })

canonical = canonical.merge(sequence_features, on="customer_id", how="left")
for flag, prefix in [
    ("engaged_flag","engagement"),
    ("read_flag","read"),
    ("reply_flag","reply"),
    ("click_flag","click"),
    ("payment_flag","payment"),
]:
    canonical = canonical.merge(first_event_table(flag, prefix), on="customer_id", how="left")

for prefix in ["engagement","read","reply","click","payment"]:
    col = f"first_{prefix}_send_at"
    canonical[f"days_from_first_send_to_first_{prefix}"] = (
        canonical[col].dt.normalize() - canonical["first_send_at"].dt.normalize()
    ).dt.days

canonical["paid_after_any_engagement_flag"] = (
    canonical["any_payment_72h"].eq(1) & canonical["n_engaged"].gt(0)
).astype(int)
canonical["paid_without_any_engagement_flag"] = (
    canonical["any_payment_72h"].eq(1) & canonical["n_engaged"].eq(0)
).astype(int)

### Early vs late journey

This is a descriptive contact-fatigue / journey-deterioration view. It is **not** causal: later attempts are selected by survival in Collections and prior non-resolution.

In [12]:
early_late = (
    x.groupby(["customer_id","journey_half"])
     .agg(
         sends=("message_id","size"),
         delivery_rate=("delivered_flag","mean"),
         engagement_rate=("engaged_flag","mean"),
         payment_association_rate=("payment_flag","mean"),
         avg_pressure14d=("n_msgs_last_14d","mean"),
     )
     .unstack("journey_half")
)

early_late.columns = [f"{metric}__{half}" for metric, half in early_late.columns]
early_late = early_late.reset_index()

canonical = canonical.merge(early_late, on="customer_id", how="left")

for metric in ["delivery_rate","engagement_rate","payment_association_rate","avg_pressure14d"]:
    e, l = f"{metric}__early", f"{metric}__late"
    if e in canonical and l in canonical:
        canonical[f"{metric}__late_minus_early"] = canonical[l] - canonical[e]

## 4.6 Final Collections segments / interpretable buckets

These features are deliberately transparent and useful for bivariate analysis and rule-candidate discovery.

In [13]:
canonical["salary_segment"] = pd.cut(
    canonical["monthly_salary_brl"],
    bins=[-np.inf,1500,2500,4000,6000,np.inf],
    labels=["le_1500","1501_2500","2501_4000","4001_6000","gt_6000"]
)
canonical["initial_balance_segment"] = pd.cut(
    canonical["initial_balance_brl"],
    bins=[-np.inf,500,1000,1500,2000,np.inf],
    labels=["le_500","501_1000","1001_1500","1501_2000","gt_2000"]
)
canonical["initial_dpd_segment"] = pd.cut(
    canonical["initial_dpd"], [0,7,15,30,60,np.inf],
    labels=["01_07","08_15","16_30","31_60","61_plus"]
)
canonical["last_dpd_segment"] = pd.cut(
    canonical["last_dpd"], [0,7,15,30,60,np.inf],
    labels=["01_07","08_15","16_30","31_60","61_plus"]
)
canonical["initial_app_recency_segment"] = pd.cut(
    canonical["initial_days_since_last_app_login"],
    [-np.inf,3,7,14,30,60,np.inf],
    labels=["00_03","04_07","08_14","15_30","31_60","61_plus"]
)
canonical["prior_transactions_segment"] = pd.cut(
    canonical["n_prior_transactions"],
    [-np.inf,0,1,3,5,np.inf],
    labels=["0","1","2_3","4_5","6_plus"]
)
canonical["account_age_segment"] = pd.cut(
    canonical["account_age_months"],
    [-np.inf,3,6,12,24,np.inf],
    labels=["00_03","04_06","07_12","13_24","25_plus"]
)
canonical["balance_to_salary_segment"] = pd.cut(
    canonical["initial_balance_to_salary"],
    [-np.inf,.10,.25,.50,.75,1,np.inf],
    labels=["le_10pct","10_25pct","25_50pct","50_75pct","75_100pct","gt_100pct"]
)

## 4.7 Reconciliation and canonical grain assertions

In [14]:
assert canonical["customer_id"].is_unique, "Canonical customer_id is not unique."
assert len(canonical) == whatsapp["customer_id"].nunique(), "Customer count changed during canonicalization."

reconciliation = pd.DataFrame({
    "metric": [
        "source_rows",
        "source_unique_messages",
        "source_unique_customers",
        "canonical_rows",
        "canonical_unique_customers",
        "canonical_duplicate_customer_ids",
        "source_total_send_cost_brl",
        "canonical_total_send_cost_brl",
        "source_failed_sends",
        "canonical_failed_sends",
        "source_attributed_payment_72h_brl",
        "canonical_attributed_payment_72h_brl",
        "customers_attributed_payment_gt_initial_balance",
    ],
    "value": [
        len(whatsapp),
        whatsapp.message_id.nunique(),
        whatsapp.customer_id.nunique(),
        len(canonical),
        canonical.customer_id.nunique(),
        canonical.customer_id.duplicated().sum(),
        len(whatsapp) * 1.0,
        canonical.message_cost_brl.sum(),
        (whatsapp.delivery_status != "delivered").sum(),
        canonical.n_failed.sum(),
        whatsapp.amount_paid_brl.sum(),
        canonical.total_attributed_payment_72h_brl.sum(),
        canonical.attributed_payment_over_initial_flag.sum(),
    ]
})
display(reconciliation)

assert np.isclose(reconciliation.loc[reconciliation.metric=="source_total_send_cost_brl","value"].iloc[0],
                  reconciliation.loc[reconciliation.metric=="canonical_total_send_cost_brl","value"].iloc[0])
assert canonical.n_failed.sum() == (whatsapp.delivery_status != "delivered").sum()
assert np.isclose(canonical.total_attributed_payment_72h_brl.sum(), whatsapp.amount_paid_brl.sum())

print("✓ Grain validated: 1 row per customer")
print("Canonical rows:", len(canonical))
print("Features:", canonical.shape[1])

,metric,value
0,source_rows,75406.0
1,source_unique_messages,75406.0
2,source_unique_customers,11724.0
3,canonical_rows,11724.0
4,canonical_unique_customers,11724.0
5,canonical_duplicate_customer_ids,0.0
6,source_total_send_cost_brl,75406.0
7,canonical_total_send_cost_brl,75406.0
8,source_failed_sends,11863.0
9,canonical_failed_sends,11863.0


✓ Grain validated: 1 row per customer
Canonical rows: 11724
Features: 365


## 4.8 Feature dictionary

A lightweight analytical dictionary is generated from the canonical output. The `family` classification makes the wide table navigable.

In [15]:
def feature_family(col):
    rules = [
        ("IDENTIFIER", ["customer_id"]),
        ("PROFILE", ["salary","payday_day","prior_transactions","account_age","state_uf"]),
        ("DELINQUENCY", ["dpd","collections_start"]),
        ("EXPOSURE_AFFORDABILITY", ["balance","remaining","reduction"]),
        ("CONTACT_PRESSURE", ["n_messages","active_contact","gap_","pressure14d","messages_per"]),
        ("CONTACTABILITY", ["deliver","failed","failure","blocked","invalid","unreachable","wasted"]),
        ("ENGAGEMENT", ["interaction","read","reply","click","engag"]),
        ("TREATMENT", ["template"]),
        ("TIMING_PAYDAY", ["send_","daypart","weekday","month_period","payday"]),
        ("APP_ENGAGEMENT", ["app_recency","last_app_login"]),
        ("PAYMENT_RECOVERY", ["payment","paid","recovery"]),
        ("JOURNEY_SEQUENCE", ["sequence","first_","last_","journey","early","late","switch"]),
        ("OPERATIONAL_EFFICIENCY", ["cost","net_recovery"]),
    ]
    lc = col.lower()
    for fam, tokens in rules:
        if any(lc == t or t in lc for t in tokens):
            return fam
    return "OTHER"

feature_dictionary = pd.DataFrame({
    "feature": canonical.columns,
    "dtype": [str(canonical[c].dtype) for c in canonical.columns],
    "family": [feature_family(c) for c in canonical.columns],
})

display(feature_dictionary.groupby("family").size().rename("n_features").sort_values(ascending=False).to_frame())
display(feature_dictionary.head(30))

,n_features
family,
ENGAGEMENT,88
CONTACTABILITY,64
TIMING_PAYDAY,49
TREATMENT,46
DELINQUENCY,27
PROFILE,21
APP_ENGAGEMENT,17
EXPOSURE_AFFORDABILITY,17
CONTACT_PRESSURE,15


,feature,dtype,family
0,customer_id,object,IDENTIFIER
1,monthly_salary_brl,float64,PROFILE
2,payday_day_of_month,int64,PROFILE
3,n_prior_transactions,int64,PROFILE
4,account_age_months,int64,PROFILE
5,state_uf,object,PROFILE
6,first_send_at,datetime64[ns],TIMING_PAYDAY
7,last_send_at,datetime64[ns],TIMING_PAYDAY
8,n_messages,int64,CONTACT_PRESSURE
9,n_active_contact_days,int64,CONTACT_PRESSURE


## 4.9 Persist enriched and canonical layers

CSV is used for portability in the case repository. Sequence columns remain JSON strings. If Parquet is available in the local environment, it can be added later for more efficient typed storage.

In [16]:
interaction_path = OUTPUT_DIR / "whatsapp_interactions_enriched.csv"
canonical_path = OUTPUT_DIR / "whatsapp_customer_canonical.csv"
dictionary_path = OUTPUT_DIR / "whatsapp_customer_feature_dictionary.csv"
reconciliation_path = OUTPUT_DIR / "whatsapp_customer_canonical_reconciliation.csv"

x.to_csv(interaction_path, index=False)
canonical.to_csv(canonical_path, index=False)
feature_dictionary.to_csv(dictionary_path, index=False)
reconciliation.to_csv(reconciliation_path, index=False)

print("Saved:")
for p in [interaction_path, canonical_path, dictionary_path, reconciliation_path]:
    print(" -", p)

Saved:
 - ..\data\interim\whatsapp_interactions_enriched.csv
 - ..\data\interim\whatsapp_customer_canonical.csv
 - ..\data\interim\whatsapp_customer_feature_dictionary.csv
 - ..\data\interim\whatsapp_customer_canonical_reconciliation.csv


## 4.10 Next step — integration with September queue

The next notebook should merge the canonical historical customer layer with `collections_queue_sep2026.csv`.

Recommended flags:

- `has_whatsapp_history`;
- `present_in_sep_queue`;
- `historical_customer_vs_new`;
- `still_open_on_sep01`;
- differences between last historical state and September entry/current state.

Only **after this integration** should the customer-level bivariate Collections analysis begin.

Keep the enriched interaction layer available for send-level questions such as template, timing, delivery, engagement and journey analysis.